# Import data

In [1]:
import pandas as pd
import re

match_id = "20120129-M-Australian_Open-F-Novak_Djokovic-Rafael_Nadal"

dfAll = pd.read_csv("./data/charting-m-points-2010s.csv")

df = dfAll[dfAll["match_id"] == match_id].copy()
df = df.sort_values("Pt").set_index("Pt")

df["Gm2"] = df["Gm2"].astype(int)

df.head()

,match_id,Set1,Set2,Gm1,Gm2,Pts,Gm#,TbSet,Svr,1st,2nd,Notes,PtWinner
Pt,,,,,,,,,,,,,
1,20120129-M-Australian_Open-F-Novak_Djokovic-Ra...,0,0,0,0,0-0,1,True,1,6w,4b27b3f3f1b2f3f1d@,NaN,1
2,20120129-M-Australian_Open-F-Novak_Djokovic-Ra...,0,0,0,0,15-0,1,True,1,6w,4f27b3n@,NaN,2
3,20120129-M-Australian_Open-F-Novak_Djokovic-Ra...,0,0,0,0,15-15,1,True,1,6w,4b28f3f3b3f3b3f2f3f1f1b2f1n@,NaN,2
4,20120129-M-Australian_Open-F-Novak_Djokovic-Ra...,0,0,0,0,15-30,1,True,1,6b28f3*,NaN,NaN,1
5,20120129-M-Australian_Open-F-Novak_Djokovic-Ra...,0,0,0,0,30-30,1,True,1,4w,5b39b1b3b3f1f2f2d@,NaN,1


# Verifying tactics
Let us verify if my perceived tactics of both players are correct.
1. Djokovic: "My backhand is better than yours."
    Djokovic's main tactic was to send a deep backhand down the line (from the ad-court) to Nadal's backhand (on the deuce-court).
2. Nadal: "My forehand is better than yours."
    Nadal, on the other hand, was trying to send his forhand inside-out crosscourt (from the deuce-court) onto Djokovic's forehand to create angles.

## Separate rally from serve

In [2]:
players = {
    1: "Novak Djokovic",   # server code 1
    2: "Rafael Nadal"
}
handed = {
    "Novak Djokovic": "R",
    "Rafael Nadal": "L"
}

SHOT_LETTERS = set("fbrsvzopuylmhijktq")

# Split each point into serve code and rally code
def SplitServeFromRally(row):
    sequence = row["2nd"] if isinstance(row["2nd"], str) and row["2nd"].strip() else row["1st"]
    # Handle missing data
    if not isinstance(sequence, str):
        return "", ""
    for idx, char in enumerate(sequence):
        if char.lower() in SHOT_LETTERS:
            return sequence[:idx], sequence[idx:]
    # No rally shot found -> serve-only point (ace, double fault, unreturnable, etc.)
    return sequence, ""
    
df[["serve", "rally"]] = df.apply(SplitServeFromRally, axis=1, result_type="expand")

df[['rally']].head()

,rally
Pt,
1,b27b3f3f1b2f3f1d@
2,f27b3n@
3,b28f3f3b3f3b3f2f3f1f1b2f1n@
4,b28f3*
5,b39b1b3b3f1f2f2d@


## Separate shots

In [3]:
POSITION_FLAGS = "+-^=;"  # approach, net, baseline smash, drop-volley, net-cord markers

def ParseToken(token: str) -> dict:
    shot = token[0].lower()
    rest = token[1:]

    positions = ""
    # Handle cases like b+;3 (approach + net-cord)
    while rest and rest[0] in POSITION_FLAGS:
        positions += rest[0]
        rest = rest[1:]

    direction = int(rest[0]) if rest and rest[0] in "0123" else None
    if direction is not None:
        rest = rest[1:]

    error_code = rest[0] if rest and rest[0] in "nwdxge!" else None
    if error_code:
        rest = rest[1:]

    outcome = rest[0] if rest and rest[0] in "*@#" else None

    return {
        "stroke": shot,
        "positions": positions,   # e.g., "+", "-", etc.
        "direction": direction,   # 1/2/3 (0 or None if missing)
        "error_code": error_code,
        "outcome": outcome,
    }

In [4]:
# Tokens: each shot starts with a letter and consumes all characters until the next letter
TOKENS_REGEX = re.compile(r"[A-Za-z][^A-Za-z]*")

# Iterate through all points
shotRows = []

for pt, row in df.iterrows():
    rally = row["rally"]
    if not rally:
        continue  # ace, double fault, etc.

    tokens = TOKENS_REGEX.findall(rally)
    server = players[row["Svr"]]
    returner = players[1 if row["Svr"] == 2 else 2]

    for idx, token in enumerate(tokens):
        hitter = returner if idx % 2 == 0 else server
        parsed = ParseToken(token)
        shotRows.append({
            "Pt": pt,
            "shotIndex": idx,
            "server": server,
            "hitter": hitter,
            **parsed
        })

shots = pd.DataFrame(shotRows)

In [5]:
def direction_label(direction, hitter):
    if direction is None:
        return "unknown"
    opponent = players[1] if hitter == players[2] else players[2]
    if handed[opponent] == "R":
        mapping = {1: "to opponent FH", 2: "middle", 3: "to opponent BH"}
    else:
        mapping = {1: "to opponent BH", 2: "middle", 3: "to opponent FH"}
    return mapping.get(direction, "unknown")



shots["targetZone"] = shots.apply(
    lambda r: direction_label(r["direction"], r["hitter"]),
    axis=1
)

# Verifying Tactics: Analysis

In [9]:
# Type 1: Djokovic backhand DTL → Nadal answer
djoko_bh_dtl = shots[(shots.hitter == "Novak Djokovic") &
                     (shots.stroke == "b") &
                     (shots.direction == 1)].copy()
djoko_bh_dtl["nextIndex"] = djoko_bh_dtl["shotIndex"] + 1

responses_type1 = djoko_bh_dtl.merge(
    shots,
    left_on=["Pt", "nextIndex"],
    right_on=["Pt", "shotIndex"],
    suffixes=("_djoko", "_nadal")
)
responses_type1 = responses_type1[responses_type1["hitter"] == "Rafael Nadal"]

# Nadal’s reply direction & court position
resp_dir_counts = responses_type1["targetZone_nadal"].value_counts()
resp_position_counts = responses_type1["positions_nadal"].value_counts()
pts_type1 = sorted(responses_type1["Pt"].unique())


# Djokovic BH DTL → Nadal response
resp_dir_counts = (
    responses_type1["targetZone_nadal"]
    .value_counts(dropna=False)
    .rename_axis("direction")
    .to_frame("count")
)
resp_dir_counts["pct"] = resp_dir_counts["count"] / resp_dir_counts["count"].sum() * 100

KeyError: 'hitter'

In [ ]:
# Type 2: Nadal FH inside-out (direction 1) → Djokovic’s reply
nadal_inside_out = shots[(shots.hitter == "Rafael Nadal") &
                         (shots.stroke == "f") &
                         (shots.direction == 1)].copy()
nadal_inside_out["nextIndex"] = nadal_inside_out["shotIndex"] + 1

responses_type2 = nadal_inside_out.merge(
    shots,
    left_on=["Pt", "nextIndex"],
    right_on=["Pt", "shotIndex"],
    suffixes=("_nadal", "_djoko")
)
responses_type2 = responses_type2[responses_type2["hitter"] == "Novak Djokovic"]

djoko_resp_dir_counts = responses_type2["targetZone_djoko"].value_counts()
djoko_resp_position_counts = responses_type2["positions_djoko"].value_counts()
pts_type2 = sorted(responses_type2["Pt"].unique())


# Nadal FH inside-out → Djokovic response
djoko_resp_dir_counts = (
    responses_type2["targetZone_djoko"]
    .value_counts(dropna=False)
    .rename_axis("direction")
    .to_frame("count")
)
djoko_resp_dir_counts["pct"] = (
    djoko_resp_dir_counts["count"] / djoko_resp_dir_counts["count"].sum() * 100
)